# Informe diario por canal de venta — mes anterior vs mes en curso

Genera **8 PowerPoint**: uno por región (Región Oriente y Región Occidente) y canal de venta (Institucional, Amas de
casa, Modernos y Tradicional), cada uno con las láminas 1, 2, 3, 4, 5 y 14 del informe de avance, con toda la
información filtrada por ese canal de venta:

| Lámina | Contenido |
|---|---|
| 1 | Portada |
| 2 | Panorama: tabla de indicadores (con las devoluciones aparte) |
| 3 | Tendencia día a día: precio de lista y neto por kg; venta neta y % de cumplimiento |
| 4 | Rangos de % de cumplimiento del precio de lista (semáforo) |
| 5 | Resumen por canal de distribución |
| 6 | Resumen por grupo de material 6 |

Ejecute todo en orden (*Run All*): siempre va **a corte de ayer** contra el mes anterior completo. Consulta la
base con `consultas.descargar_mes()` (la misma consulta del reporte semanal) y procesa en memoria, sin Parquet. El
precio de lista y los % se calculan sin las devoluciones.

# A. Preparación
## A.1 Parámetros

> **Siempre a corte de ayer** contra el mes anterior completo (A.4). `CANALES_VENTA_INFORME`: los canales de
> venta con PowerPoint.

In [1]:
from datetime import date, timedelta
from pathlib import Path

AQUI = Path.cwd()
SALIDA = AQUI / "output" / "Informe_Diario_Canal_Venta"   # carpeta donde se guardan los resultados de este notebook
SALIDA.mkdir(parents=True, exist_ok=True)
CANALES_VENTA_INFORME = ["Institucional", "Amas de casa", "Modernos", "Tradicional"]   # un PowerPoint por canal
TOP_RANGOS = 8                        # grupos de material 6 en la tabla resumen


## A.2 Librerías y conexión

Usa la conexión `DB_*` del `.env` de esta carpeta o, si no hay, el del proyecto de lista de precios.

In [2]:
import base64
import io
import os
from contextlib import contextmanager

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display
from sqlalchemy import URL, create_engine, text

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


def engine():
    load_dotenv(AQUI / ".env") or load_dotenv(AQUI.parent / "Lista de precios" / ".env")
    faltan = [v for v in ("DB_HOST", "DB_NAME", "DB_USER", "DB_PASSWORD") if not os.getenv(v)]
    if faltan:
        raise RuntimeError(f"Faltan en el .env: {', '.join(faltan)}. "
                           f"Copie .env.example como .env en {AQUI} y completelo.")
    url = URL.create("postgresql+psycopg2", username=os.environ["DB_USER"], password=os.environ["DB_PASSWORD"],
                     host=os.environ["DB_HOST"], port=int(os.getenv("DB_PORT", "5432")),
                     database=os.environ["DB_NAME"])
    return create_engine(url, pool_pre_ping=True)


ENGINE = engine()
with ENGINE.connect() as c:
    c.execute(text("SELECT 1"))
print(f"Conexion OK: {ENGINE.url.host}/{ENGINE.url.database}")

Conexion OK: 192.168.13.93/commercial_bi


## A.3 Consulta (consultas.py)

La consulta y los cálculos por fila están en `consultas.py` (`SQL_REPORTES`, `calcular()`, `descargar_mes()`), que
comparten el reporte diario y el semanal:

- **Ventas sin devoluciones:** kg, venta neta, bruta y descuentos salen solo de las líneas con `value >= 0`. Las
  devoluciones (líneas con `value < 0`, registradas a precio neto y sin descuento) van aparte en `Devolucion Kg`
  y `Devolucion Valor`, y no entran al precio de lista ni al % de cumplimiento.
- `value` es la venta **neta**; bruta = `value + discount_value`. `discount_value` es el ajuste total.
- Fecha corta = `short_dated_discount`; ajuste de precio de mercado = `discount_value − short_dated_discount`.
- Grano: fecha × cliente × grupo de material 6 × distribuidora × canal × sector. El % de lista y el rango (22
  tramos) se calculan sobre ese grano.

In [3]:
# El SQL, los calculos (% de lista, rangos, motivo) y la descarga estan en consultas.py: los comparten el reporte
# diario y el semanal.
from consultas import FILAS_POR_PARTE, _ROTULO_SIN_BRUTA, _ROTULOS, descargar_mes


## A.4 Periodo

Del 1 del mes anterior (completo) hasta `CORTE_D` (incluido): en el reporte diario, **siempre hasta ayer**.

In [4]:
CORTE_D = date.today() - timedelta(days=1)                       # el diario siempre va hasta ayer
INICIO_ACT = CORTE_D.replace(day=1)                              # 1 del mes en curso
INICIO_ANT = (INICIO_ACT - timedelta(days=1)).replace(day=1)     # 1 del mes anterior
FIN_D = CORTE_D + timedelta(days=1)                              # EXCLUSIVO
print(f"Periodo: {INICIO_ANT:%d/%m/%Y} (mes anterior completo) a {CORTE_D:%d/%m/%Y} (incluido)")


Periodo: 01/09/2026 (mes anterior completo) a 22/10/2026 (incluido)


## A.5 Nombres, formato y funciones comunes

In [5]:
SOCIEDADES = {"1000": "Región Oriente", "2000": "Región Occidente"}   # company_code: 1000 Región Oriente, 2000 Región Occidente
VISTAS = {"Región Oriente": "Región Oriente", "Región Occidente": "Región Occidente"}
ARCHIVO = {"Región Oriente": "Region_Oriente", "Región Occidente": "Region_Occidente"}   # nombre en los archivos
CORTO = {soc: vista for vista, soc in VISTAS.items()}
NOMBRE_CANAL_VENTA = {"Directo": "Amas de casa"}
# Dimensiones de los rangos (secciones 3 a 6 y 8): rotulo -> columna del Parquet (None = el total)
DIMENSIONES = {"Total": None,
               "Canal de distribución": "Canal de Distribucion",
               "Grupo de material 6": "Nombre GR6"}
TOPE = {"Grupo de material 6": TOP_RANGOS, "Distribuidora": TOP_RANGOS}
CON_CALOR = {"Grupo de material 6"}   # tablas resumen con mapa de calor en los % de ajuste
MEDIDAS = ["Venta Kg", "Venta Bruta", "Venta Neta",
           "Descuento Ajuste Precio Mercado", "Descuento Rotacion / Fecha Corta",
           "Devolucion Kg", "Devolucion Valor"]   # ventas sin devoluciones; las devoluciones, aparte

# --- nombres normalizados
VENTA_BRUTA, VENTA_NETA, VENTA_KG = "Venta bruta ($)", "Venta neta ($)", "Venta kg"
PRECIO_LISTA, PRECIO_NETO = "Precio lista por kg", "Precio neto por kg"
L_GEN, D_GEN = "% cumplimiento del precio de lista general", "% Ajuste general"
L_AJ, D_AJ = "% cumplimiento del precio de lista ajuste de precio de mercado", "% Ajuste de precio de mercado"
L_FC, D_FC = "% cumplimiento del precio de lista fecha corta", "% Ajuste de fecha corta"
P_GEN, P_AJ, P_FC = "Ajuste general ($)", "Ajuste de precio de mercado ($)", "Ajuste de fecha corta ($)"
MILLON = 1     # divisor de los montos: 1 = pesos completos (1e6 = millones)
PILAR = [L_GEN, D_GEN, L_AJ, D_AJ, L_FC, D_FC]
COMPACTO = [VENTA_KG, VENTA_BRUTA, P_GEN, VENTA_NETA, PRECIO_NETO, *PILAR]

# --- colores: motivo y tipo de ajuste comparten color (ajuste = verde, fecha corta = amarillo)
MOTIVOS = {"Con fecha corta": "#eda100", "Solo ajuste de precio de mercado": "#1baf7a", "Sin ajuste": "#b5b4af"}
COLOR_SOC = {"Región Oriente": "#2a78d6", "Región Occidente": "#eb6834"}
TINTE_SOC = {"Región Oriente": "#a9c9f0", "Región Occidente": "#f5b49a"}

TRAMOS = [r[4:] for r in _ROTULOS]  # "100% O MAS", ..., "Menos de 50%"
MESES_ES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
            "septiembre", "octubre", "noviembre", "diciembre"]


# --- formato colombiano
def num(v, dec: int = 1) -> str:
    """Miles con punto, decimales con coma (3.831,3)."""
    if pd.isna(v):
        return "-"
    return f"{v:,.{dec}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def pct(v) -> str:
    return "-" if pd.isna(v) else num(v) + "%"


def pesos(v) -> str:
    return "-" if pd.isna(v) else "$" + num(v, 0)


def variacion(v, en_puntos: bool) -> str:
    """+5,9% (variacion) o -6,4 pts (diferencia entre porcentajes)."""
    if pd.isna(v):
        return "-"
    return ("+" if v >= 0 else "-") + num(abs(v)) + (" pts" if en_puntos else "%")


def formato_de(indicador: str):
    if indicador.startswith("%"):
        return pct
    if indicador.startswith("Venta kg"):   # kilos completos, sin decimales
        return lambda v: num(v, 0)
    if indicador.startswith("Precio") or indicador.endswith("($)"):
        return pesos
    return num


# --- indicadores (todo desde sumas, nunca promediando % de filas)
def indicadores(s: pd.DataFrame) -> pd.DataFrame:
    """s: sumas de MEDIDAS (cualquier indice) -> BLOQUE de indicadores."""
    bruta = s["Venta Bruta"]
    b = bruta.where(bruta != 0)
    kg = s["Venta Kg"].where(s["Venta Kg"] != 0)
    aj, fc = s["Descuento Ajuste Precio Mercado"], s["Descuento Rotacion / Fecha Corta"]
    gen = aj + fc
    return pd.DataFrame({
        VENTA_BRUTA: bruta / MILLON,
        VENTA_NETA: s["Venta Neta"] / MILLON,
        VENTA_KG: s["Venta Kg"],
        PRECIO_LISTA: bruta / kg,
        PRECIO_NETO: s["Venta Neta"] / kg,
        L_GEN: (bruta - gen) / b * 100,
        D_GEN: gen / b * 100,
        P_GEN: gen / MILLON,
        L_AJ: (bruta - aj) / b * 100,
        D_AJ: aj / b * 100,
        P_AJ: aj / MILLON,
        L_FC: (bruta - fc) / b * 100,
        D_FC: fc / b * 100,
        P_FC: fc / MILLON,
    }, index=s.index)


def comparar(t: pd.DataFrame) -> pd.DataFrame:
    """t: filas = años -> filas = indicadores; columnas 2025, 2026 y Variación (% o pts)."""
    t = t.T
    out = t[[A1, A2]].copy()
    out.columns = [str(A1), str(A2)]
    es_pct = out.index.str.startswith("%")
    out["Variación"] = ((out[str(A2)] / out[str(A1)] - 1) * 100).where(~es_pct, out[str(A2)] - out[str(A1)])
    return out


def es_variacion(col) -> bool:
    """Columna de variación entre años: 'Variación' en cualquier nivel del encabezado."""
    return any(str(x).startswith("Variación") for x in (col if isinstance(col, tuple) else (col,)))


def ejes_limpios(ax, formato_y=None, formato_x=None):
    ax.grid(axis="y" if formato_y else "x", color="#e6e5e1", linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    ax.tick_params(colors="#52514e", labelsize=8)
    if formato_y:
        ax.yaxis.set_major_formatter(formato_y)
    if formato_x:
        ax.xaxis.set_major_formatter(formato_x)

## A.6 Consulta y lectura del mes

Consulta la base con `descargar_mes()` (un DataFrame en memoria, sin Parquet) y lo recorre por partes dejando las tablas del mes con el canal de venta: `diario_cv` (sociedad × canal de venta ×
día) y `cubo_cv` (dimensión × miembro × sociedad × canal de venta × periodo × rango × motivo). `diario_soc` y
`cubo_d` son los mismos con todos los canales juntos; `solo_canal_venta()` (C.8) los cambia por los de un canal.

In [6]:
columnas = ["Fecha", "Distribuidora", *MEDIDAS, "Rango Total", "Canal de Venta",
            *[c for c in DIMENSIONES.values() if c]]
print(f"Consultando {INICIO_ANT} -> {FIN_D} (exclusivo)...")
datos = descargar_mes(INICIO_ANT, FIN_D, columnas=[*columnas, "Sociedad"], motor=ENGINE)
FECHA_MAX_D = datos["Fecha"].max()
MES_ANT_TXT, MES_ACT_TXT = MESES_ES[INICIO_ANT.month - 1], MESES_ES[INICIO_ACT.month - 1]
ETQ_ANT = f"{MES_ANT_TXT[:3].capitalize()} {INICIO_ANT.year}"          # "Sep 2026"
ETQ_ACT = f"{MES_ACT_TXT[:3].capitalize()} al {FECHA_MAX_D.day}"       # "Oct al 13"
PERIODOS_D = [ETQ_ANT, ETQ_ACT]


def juntar(partes: list) -> pd.DataFrame:
    t = pd.concat(partes)
    return t.groupby(level=list(range(t.index.nlevels))).sum()


partes_dia, partes_cubo = [], []
for inicio in range(0, len(datos), FILAS_POR_PARTE):   # por partes, como se leia el Parquet
    d = datos.iloc[inicio:inicio + FILAS_POR_PARTE].copy()
    d["Sociedad"] = d["Sociedad"].map(SOCIEDADES).fillna(d["Sociedad"]).fillna("Sin sociedad")
    d = d[d["Sociedad"].isin(VISTAS.values())]
    d["Canal de Venta"] = d["Canal de Venta"].replace(NOMBRE_CANAL_VENTA).fillna("(sin canal)")
    d["Dia"] = d["Fecha"].dt.to_period("D")
    partes_dia.append(d.groupby(["Sociedad", "Canal de Venta", "Dia"])[MEDIDAS].sum())

    # el nivel "Año" lleva el periodo: mes anterior cerrado o mes en curso
    d["Año"] = pd.Series(ETQ_ANT, index=d.index).mask(d["Fecha"] >= pd.Timestamp(INICIO_ACT), ETQ_ACT)
    d["Motivo"] = (pd.Series("Sin ajuste", index=d.index)
                   .mask(d["Descuento Ajuste Precio Mercado"] > 0, "Solo ajuste de precio de mercado")
                   .mask(d["Descuento Rotacion / Fecha Corta"] > 0, "Con fecha corta"))
    for dim, col in DIMENSIONES.items():
        miembro = (d[col].fillna("(vacío)") if col else pd.Series("Total", index=d.index)).rename("Miembro")
        partes_cubo.append(pd.concat(
            {dim: d.groupby([miembro, "Sociedad", "Canal de Venta", "Año", "Rango Total", "Motivo"])[MEDIDAS].sum()},
            names=["Dimension"]))

diario_cv = juntar(partes_dia)            # sociedad x canal de venta x dia
cubo_cv = juntar(partes_cubo)             # dimension x miembro x sociedad x canal de venta x periodo x rango x motivo
# todos los canales juntos; solo_canal_venta() las cambia por las de un canal
diario_soc = diario_cv.groupby(level=["Sociedad", "Dia"]).sum()
cubo_d = cubo_cv.groupby(level=[n for n in cubo_cv.index.names if n != "Canal de Venta"]).sum()
del partes_dia, partes_cubo, datos
print(f"Datos hasta {FECHA_MAX_D:%d/%m/%Y} · {ETQ_ANT} vs {ETQ_ACT} | canales de venta: "
      f"{', '.join(diario_cv.index.unique('Canal de Venta'))}")

Consultando 2026-09-01 -> 2026-10-23 (exclusivo)...
  500,000 filas
  707,761 filas
Datos hasta 22/10/2026 · Sep 2026 vs Oct al 22 | canales de venta: Amas de casa, Institucional, Modernos, Tradicional


# B. Funciones de las láminas

Solo las definiciones que usan las láminas, con la numeración de secciones del informe general. Todas corren
dentro de `modo_corte()` (sección 10) y, en este notebook, de `solo_canal_venta()` (C.8). Los % se calculan siempre
desde las sumas y sin las devoluciones.

## 3. Rangos de % cumplimiento del precio de lista: definiciones

In [7]:
def cubo_de(dim: str, soc, miembro=None) -> pd.DataFrame:
    c = cubo.xs(dim, level="Dimension")
    c = c[c.index.get_level_values("Sociedad") == soc]
    if miembro is not None:
        c = c[c.index.get_level_values("Miembro") == miembro]
    return c


def con_rango(c: pd.DataFrame) -> pd.DataFrame:
    """Solo las ventas con venta bruta positiva (las que tienen rango)."""
    return c[c.index.get_level_values("Rango Total") != _ROTULO_SIN_BRUTA]


def miembros(dim: str, soc) -> list:
    """Miembros con venta bruta en el ultimo año, de mayor a menor (top N si aplica)."""
    b = cubo_de(dim, soc)
    b = b[b.index.get_level_values("Año") == A2]["Venta Bruta"].groupby(level="Miembro").sum()
    b = b[b > 0].sort_values(ascending=False)
    return list(b.index[:TOPE.get(dim, len(b))])


def resumen_dim(dim: str, soc, columnas=COMPACTO) -> pd.DataFrame:
    """Miembro x (indicador, año)."""
    s = cubo_de(dim, soc).groupby(level=["Miembro", "Año"]).sum()
    t = indicadores(s)[columnas].unstack("Año")
    t.columns = pd.MultiIndex.from_tuples([(i, str(a_)) for i, a_ in t.columns])
    t = t.reindex(columns=pd.MultiIndex.from_product([columnas, [str(A1), str(A2)]]))
    t.index.name = dim
    return t.reindex(miembros(dim, soc))


def rango_principal(c: pd.DataFrame, anio: int):
    """Rango con mas venta bruta entre las ventas CON ajuste (sin el rango 100% o mas). None si no hay venta."""
    r = con_rango(c)
    r = r[(r.index.get_level_values("Año") == anio)
          & (r.index.get_level_values("Rango Total") != _ROTULOS[0])]["Venta Bruta"].groupby(level="Rango Total").sum()
    r = r[r > 0]
    return r.idxmax()[4:] if len(r) else None


def frase_rango(c: pd.DataFrame) -> str:
    """'la venta con ajuste pasó del rango X al Y' / 'sigue concentrada en X'."""
    r1, r2 = rango_principal(c, A1), rango_principal(c, A2)
    if r2 is None:
        return f"sin venta con ajuste en {A2}"
    if r1 is None:
        return f"la venta con ajuste se concentra en “{r2}” (sin venta en {A1})"
    if r1 == r2:
        return f"la venta con ajuste sigue concentrada en “{r2}”"
    return f"la venta con ajuste pasó del rango “{r1}” al “{r2}”"


SIGLA_MOTIVO = {"Con fecha corta": "FC", "Solo ajuste de precio de mercado": "AM", "Sin ajuste": "SA"}


def pct_fino(v) -> str:
    """% con un decimal; los menores a 0,1% con dos y los menores a 0,01% como "<0,01%", para que no salgan en 0."""
    if 0 < v < 0.01:
        return "<0,01%"
    return num(v, 2) + "%" if 0 < v < 0.1 else pct(v)


def figura_rangos(c: pd.DataFrame, titulo: str, figsize=(13, 7.4), detalle=False):
    """22 rangos (filas) x motivo, un panel por año. Largo = % de la venta bruta del año; al lado, % y ajuste general.
    detalle: al final de la etiqueta, el % de la venta bruta de cada motivo (siglas de SIGLA_MOTIVO, en la leyenda)."""
    cr = con_rango(c)
    r = cr["Venta Bruta"]
    por_motivo = r.groupby(level=["Año", "Rango Total", "Motivo"]).sum()
    dcto = cr["Descuento Ajuste Precio Mercado"] + cr["Descuento Rotacion / Fecha Corta"]
    por_rango = dcto.groupby(level=["Año", "Rango Total"]).sum()
    total_anio = r.groupby(level="Año").sum()
    fig, ejes = plt.subplots(1, len(ANIOS), figsize=figsize, sharey=True, sharex=True)
    tope = 0
    for ax, anio in zip(ejes, ANIOS):
        if anio not in total_anio.index:
            ax.set_title(f"{anio}: sin venta", loc="left", fontsize=11)
            continue
        t = (por_motivo[por_motivo.index.get_level_values("Año") == anio].droplevel("Año").unstack("Motivo")
             .reindex(_ROTULOS).fillna(0) / total_anio[anio] * 100)
        pesos_ = por_rango[por_rango.index.get_level_values("Año") == anio].droplevel("Año").reindex(_ROTULOS).fillna(0)
        base = pd.Series(0.0, index=t.index)
        for m, color in MOTIVOS.items():
            v = t[m] if m in t else pd.Series(0.0, index=t.index)
            ax.barh(TRAMOS, v, left=base, color=color, edgecolor="white", linewidth=1, height=0.75,
                    label=f"{m} ({SIGLA_MOTIVO[m]})" if detalle else m)
            base += v
        for i, (total_, plata) in enumerate(zip(base, pesos_)):
            if total_ > 0:   # todas las barras con venta, aunque sean muy pequeñas
                etiqueta = f"{pct_fino(total_)} · ajuste {pesos(plata)}"
                if detalle:
                    etiqueta += "".join(f" · {s_} {pct_fino(t[m].iloc[i])}" for m, s_ in SIGLA_MOTIVO.items()
                                        if m in t and t[m].iloc[i] > 0)
                ax.annotate(etiqueta, (total_, i), xytext=(3, 0),
                            textcoords="offset points", va="center", fontsize=6.5, color="#52514e")
        tope = max(tope, base.max())
        ax.set_title(str(anio), loc="left", fontsize=11)
        ejes_limpios(ax, formato_x=lambda v, _: f"{v:,.0f}%")
    for ax in ejes:
        ax.set_xlim(0, tope * (3.0 if detalle else 2.1))  # espacio para la etiqueta del ajuste
    ejes[0].invert_yaxis()
    ejes[-1].legend(loc="lower right", frameon=False, fontsize=8)
    fig.suptitle(titulo, x=0.01, ha="left", fontsize=11)
    plt.tight_layout()
    return fig

## 10. Avance del mes en curso: definiciones

In [8]:
DIAS_EJE = pd.period_range(INICIO_ANT, FECHA_MAX_D, freq="D")
KG_DIA, BRUTA_DIA = "Venta kg por día", "Venta bruta por día ($)"
KG_DEV, VAL_DEV = "Venta kg devuelta", "Venta devuelta ($)"


def diario_de(soc) -> pd.DataFrame:
    """Sumas por dia de la sociedad (en el modo de corte reemplaza a mensual_de)."""
    return diario_soc[diario_soc.index.get_level_values("Sociedad") == soc].groupby(level="Dia").sum()


def dias_con_venta(soc) -> pd.Series:
    """Dias con venta bruta en cada periodo."""
    dd = diario_de(soc)
    dd = dd[dd["Venta Bruta"] > 0]
    return pd.Series(dd.index.to_timestamp() >= pd.Timestamp(INICIO_ACT), index=dd.index).map(
        {False: ETQ_ANT, True: ETQ_ACT}).value_counts().reindex(PERIODOS_D, fill_value=0)


def panorama_corte(soc) -> pd.DataFrame:
    """La tabla del panorama (A1 = mes anterior, A2 = mes en curso) mas la venta por dia con venta."""
    s = cubo_d.xs("Total", level="Dimension")
    s = s[s.index.get_level_values("Sociedad") == soc].groupby(level="Año").sum().reindex(PERIODOS_D)
    ind = indicadores(s)
    ind[KG_DEV] = s["Devolucion Kg"]                 # devoluciones aparte (no entran a los precios ni a los %)
    ind[VAL_DEV] = s["Devolucion Valor"] / MILLON
    ind = ind[[VENTA_BRUTA, VENTA_NETA, VENTA_KG, KG_DEV, VAL_DEV,
               *[c_ for c_ in ind.columns if c_ not in (VENTA_BRUTA, VENTA_NETA, VENTA_KG, KG_DEV, VAL_DEV)]]]
    dias = dias_con_venta(soc)
    ind[KG_DIA] = ind[VENTA_KG] / dias
    ind[BRUTA_DIA] = ind[VENTA_BRUTA] / dias
    return comparar(ind)


CAMBIOS_CORTE = {
    "A1": ETQ_ANT, "A2": ETQ_ACT, "ANIOS": PERIODOS_D, "PERIODO": "mes cerrado vs en curso ·",
    "MES_FIN_TXT": f"{MES_ACT_TXT} al {FECHA_MAX_D.day}", "MES_TXT": ETQ_ACT, "FECHA_MAX": FECHA_MAX_D,
    "cubo": cubo_d,   # solo_canal_venta() lo cambia por el del canal
    # evolucion dia a dia (seccion 2)
    "mensual_de": diario_de, "MESES_EJE": DIAS_EJE, "X_MES": list(range(len(DIAS_EJE))),
    "ETIQ_MES": DIAS_EJE.strftime("%d/%m"), "CORTE_ANIO": sum(DIAS_EJE.month == INICIO_ANT.month) - 0.5,
    "PANELES": {"kg": "Venta kg por día",
                "pesos": "Venta bruta y venta neta por día ($) — la franja es el ajuste",
                "precio": "Precio de lista y precio neto por kg, por día",
                "aplicado": "% cumplimiento del precio de lista por día (100% − % de ajuste)"},
    # rotulos del informe general (C.2) que llevan los años
    "ANIOS_CONC": {L_GEN: (ETQ_ANT, ETQ_ACT), D_FC: (ETQ_ANT, ETQ_ACT), L_FC: (ETQ_ANT, ETQ_ACT)},
    "PANORAMA": None,   # se calcula al entrar
}


@contextmanager
def modo_corte():
    """Mientras dura el `with`, las funciones de B y C comparan el mes anterior cerrado con el mes en curso; al salir
    (aunque haya error) todo queda como estaba. Los rotulos de las secciones (C.2) se toman si ya existen."""
    g = globals()
    cambios = {**CAMBIOS_CORTE, "SECCIONES": {**g.get("SECCIONES", {}), 2: f"Panorama {ETQ_ANT} vs {ETQ_ACT}"}}
    antes = {k: g[k] for k in cambios if k in g}
    g.update(cambios)
    try:
        g["PANORAMA"] = {vista: panorama_corte(soc) for vista, soc in VISTAS.items()}
        yield
    finally:
        for k in cambios:
            if k in antes:
                g[k] = antes[k]
            else:
                g.pop(k, None)

## 12. Hallazgos: definiciones

In [9]:
def v_(vista, ind, anio):
    return PANORAMA[vista].loc[ind, str(anio)]


def ch_(vista, ind):
    return PANORAMA[vista].loc[ind, "Variación"]


def sube_baja(v, plural=False):
    return ("subieron" if v >= 0 else "bajaron") if plural else ("subió" if v >= 0 else "bajó")


def de_a(vista, ind):
    return f"{formato_de(ind)(v_(vista, ind, A1))} → {formato_de(ind)(v_(vista, ind, A2))}"

# C. Salidas

`Informe_Diario_<región>_<canal de venta>_<mes anterior>_vs_<mes en curso>_al_<día>.pptx` (8 archivos), en `SALIDA`.

## C.1 Funciones comunes de PowerPoint

In [10]:
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import MSO_ANCHOR, PP_ALIGN
from pptx.util import Inches, Pt


def rgb(h: str) -> RGBColor:
    return RGBColor.from_string(h.lstrip("#").upper())


TINTA, TINTA_2, TINTA_3 = rgb("#0b0b0b"), rgb("#52514e"), rgb("#8c8b87")
FONDO_ENC, BLANCO, ACENTO = rgb("#f0efeb"), rgb("#ffffff"), rgb("#2a78d6")
ANCHO, ALTO, MARGEN = Inches(13.333), Inches(7.5), Inches(0.5)
UTIL_W = ANCHO - 2 * MARGEN
CUERPO_Y = Inches(1.6)
CUERPO_H = ALTO - CUERPO_Y - Inches(0.55)
prs = None   # presentacion en construccion


def nueva_presentacion():
    global prs
    prs = Presentation()
    prs.slide_width, prs.slide_height = ANCHO, ALTO


def guardar(nombre: str):
    archivo_ = SALIDA / nombre
    try:
        prs.save(archivo_)
        print(f"{archivo_.name}: {len(prs.slides)} láminas")
    except PermissionError:
        print(f"No se pudo guardar {archivo_.name}: ciérrelo en PowerPoint y vuelva a correr esta celda.")


def caja_texto(d, x, y, w, h, parrafos, tam=14, color=TINTA, negrita=False, alinear=PP_ALIGN.LEFT, espacio=0):
    """parrafos: texto o lista; cada parrafo es texto o lista de (texto, negrita)."""
    caja = d.shapes.add_textbox(x, y, w, h)
    tf = caja.text_frame
    tf.word_wrap = True
    for i, par in enumerate([parrafos] if isinstance(parrafos, str) else parrafos):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = alinear
        p.space_after = Pt(espacio)
        for trozo, neg in ([(par, negrita)] if isinstance(par, str) else par):
            r = p.add_run()
            r.text = trozo
            r.font.size, r.font.bold = Pt(tam), neg
            r.font.color.rgb = color
    return caja


def portada(titulo: str, linea_1: str, linea_2: str, color: str):
    """Titulo, barra de color, dos lineas de contexto y el area al pie."""
    d = prs.slides.add_slide(prs.slide_layouts[6])
    caja_texto(d, MARGEN, Inches(2.2), UTIL_W, Inches(0.9), titulo, tam=28, negrita=True)
    barra = d.shapes.add_shape(1, MARGEN, Inches(3.2), Inches(1.2), Inches(0.06))
    barra.fill.solid()
    barra.fill.fore_color.rgb = rgb(color)
    barra.line.fill.background()
    caja_texto(d, MARGEN, Inches(3.4), UTIL_W, Inches(0.4), linea_1, tam=18, color=TINTA_2)
    caja_texto(d, MARGEN, Inches(4.0), UTIL_W, Inches(0.34), linea_2, tam=14, color=TINTA_2)
    caja_texto(d, MARGEN, Inches(6.5), UTIL_W, Inches(0.4), "Sistemas de Información", tam=12, color=TINTA_3)


def nombre_sociedad(soc: str) -> str:
    """Nombre de la region para las portadas."""
    return soc


def imagen(d, fig, x=MARGEN, y=CUERPO_Y, w=UTIL_W, h=CUERPO_H):
    """Figura ajustada a la caja w x h, sin deformarla y centrada en horizontal."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=170, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    buf.seek(0)
    pic = d.shapes.add_picture(buf, x, y)
    escala = min(w / pic.width, h / pic.height)
    pic.width, pic.height = int(pic.width * escala), int(pic.height * escala)
    pic.left = int(x + (w - pic.width) / 2)
    return pic


def color_calor(k: float) -> RGBColor:
    """Escala de un solo tono: blanco (k=0) a naranja (k=1), con texto oscuro legible en todo el rango."""
    a, b = (255, 255, 255), (0xE0, 0x6A, 0x2E)
    k = 0.06 + 0.7 * max(0.0, min(1.0, k))
    return RGBColor(*[int(x + (y - x) * k) for x, y in zip(a, b)])


def tabla(d, df: pd.DataFrame, x=MARGEN, y=CUERPO_Y, w=UTIL_W, tam=10, ancho_indice=0.25,
          negrita_filas=(), alto_fila=None, calor=None, anchos=None):
    """Tabla nativa con el texto ya formateado. Columnas de 2 niveles -> 2 filas de encabezado (combinadas).
    calor: DataFrame numerico del mismo tamaño; las columnas con valores se pintan como mapa de calor. La escala es
    por columna, o por indicador (todos sus años juntos) si las columnas tienen 2 niveles.
    anchos: ancho de cada columna en pulgadas, incluida la del indice (si no, se reparten por igual)."""
    if calor is not None:
        calor = calor.astype(float).copy()
        grupos = (calor.columns.get_level_values(0) if isinstance(calor.columns, pd.MultiIndex)
                  else calor.columns)
        for g in pd.unique(grupos):
            cols_g = [col for col, gg in zip(calor.columns, grupos) if gg == g]
            bloque = calor[cols_g]
            lo, hi = bloque.min().min(), bloque.max().max()
            calor[cols_g] = (bloque - lo) / ((hi - lo) or 1)
    dos = isinstance(df.columns, pd.MultiIndex)
    n_enc = 2 if dos else 1
    n_f, n_c = df.shape[0] + n_enc, df.shape[1] + 1
    alto_fila = alto_fila or Inches(0.08 + tam * 0.021)
    forma = d.shapes.add_table(n_f, n_c, x, y, w, alto_fila * n_f)
    t = forma.table
    t.first_row, t.horz_banding = True, False
    t.columns[0].width = int(w * ancho_indice)
    for j in range(1, n_c):
        t.columns[j].width = int(w * (1 - ancho_indice) / (n_c - 1))
    for j, a_ in enumerate(anchos or []):
        t.columns[j].width = Inches(a_)
    for i in range(n_f):
        t.rows[i].height = alto_fila
        for j in range(n_c):
            celda_ = t.cell(i, j)
            if i < n_enc:
                if j == 0:
                    texto_ = (df.index.name or "") if i == n_enc - 1 else ""
                elif dos:
                    texto_ = str(df.columns[j - 1][i])
                else:
                    texto_ = str(df.columns[j - 1])
            else:
                texto_ = str(df.index[i - n_enc]) if j == 0 else str(df.iat[i - n_enc, j - 1])
            celda_.text = texto_
            celda_.fill.solid()
            celda_.fill.fore_color.rgb = FONDO_ENC if i < n_enc else BLANCO
            if calor is not None and i >= n_enc and j > 0 and pd.notna(calor.iat[i - n_enc, j - 1]):
                celda_.fill.fore_color.rgb = color_calor(calor.iat[i - n_enc, j - 1])
            celda_.vertical_anchor = MSO_ANCHOR.MIDDLE
            celda_.margin_left = celda_.margin_right = Inches(0.04)
            celda_.margin_top = celda_.margin_bottom = Inches(0.01)
            p = celda_.text_frame.paragraphs[0]
            p.alignment = PP_ALIGN.LEFT if j == 0 else (PP_ALIGN.CENTER if i < n_enc else PP_ALIGN.RIGHT)
            for r in p.runs:
                r.font.size = Pt(tam)
                r.font.bold = i < n_enc or df.index[i - n_enc] in negrita_filas
                r.font.color.rgb = TINTA
    if dos:  # combina el encabezado superior repetido
        j = 1
        while j < n_c:
            k = j
            while k + 1 < n_c and df.columns[k][0] == df.columns[j - 1][0]:
                k += 1
            if k > j:
                for jj in range(j + 1, k + 1):   # vaciar antes de combinar: merge junta los textos
                    t.cell(0, jj).text = ""
                t.cell(0, j).merge(t.cell(0, k))
            j = k + 1
    return forma


def a_texto(t: pd.DataFrame, filas_indicador=False) -> pd.DataFrame:
    """Formatea una tabla: por fila (indicadores en filas) o por columna (nivel 0)."""
    out = t.astype(object).copy()
    for c in t.columns:
        for i in t.index:
            ind = i if filas_indicador else (c[0] if isinstance(c, tuple) else c)
            v = t.at[i, c]
            if isinstance(v, str):          # columnas de texto (p. ej. canal de venta)
                continue
            out.at[i, c] = variacion(v, ind.startswith("%")) if es_variacion(c) else formato_de(ind)(v)
    return out


def corto(ind: str) -> str:
    """Rotulo corto para encabezados de tabla."""
    return ind.replace("% cumplimiento del precio de lista", "% cumplimiento").replace("de precio de mercado", "precio mercado")

## C.2 Informe general por sociedad: definiciones

In [11]:
SECCIONES = {
    1: "Contexto del alcance",
    2: f"Panorama {ETQ_ANT} vs {ETQ_ACT}",   # el mismo que pone modo_corte()
    3: "Rangos de % cumplimiento del precio de lista",
    4: "Rangos de % de lista precio · Canal de distribución",
    5: "Rangos de % de lista precio · Canal de venta",
    6: "Rangos de % de lista precio · Grupo material 6",
    7: "Canal de venta y canal de distribución",
    8: "Concentración del ajuste - Distribuidoras",
}
COLS_RES = [VENTA_KG, VENTA_BRUTA, P_GEN, *PILAR]


def sec(n: int) -> str:
    return f"{n}. {SECCIONES[n]}"


def lamina(titulo: str = None, subtitulo: str = None, seccion: str = None, compacta=False):
    """Lamina del informe general: seccion (azul, 16), titular (de 19 a 14 segun el largo, para que quepa en una
    linea), bajada y numero de lamina. compacta: todo mas arriba, para tablas largas."""
    d = prs.slides.add_slide(prs.slide_layouts[6])
    y_sec, y_tit, y_sub = (0.08, 0.41, 0.75) if compacta else (0.25, 0.62, 1.13)
    caja_texto(d, MARGEN, Inches(y_sec), UTIL_W, Inches(0.37), seccion.upper(), tam=16, color=ACENTO, negrita=True)
    if titulo:
        tam = next((t_ for t_ in (19, 16, 14) if len(titulo) <= 1770 / t_), 14)
        caja_texto(d, MARGEN, Inches(y_tit), UTIL_W, Inches(0.4), titulo, tam=16 if compacta else tam, negrita=True)
    if subtitulo:
        caja_texto(d, MARGEN, Inches(y_sub), UTIL_W, Inches(0.4), subtitulo, tam=12, color=TINTA_2)
    caja_texto(d, ANCHO - MARGEN - Inches(0.6), Inches(7.1), Inches(0.6), Inches(0.24), str(len(prs.slides)),
               tam=8, color=TINTA_3, alinear=PP_ALIGN.RIGHT)
    return d


def lamina_rangos(dim, vista, soc, m, seccion):
    """La grafica de rangos (con %, ajuste y % de cada motivo al lado de cada barra)."""
    c = cubo_de(dim, soc, m)
    nombre = vista if dim == "Total" else f"{m} ({vista})"
    ind = indicadores(c.groupby(level="Año").sum()).reindex(ANIOS)  # un miembro puede no tener venta un año
    bajada = (f"% cumplimiento del precio de lista general {pct(ind.loc[A1, L_GEN])} → {pct(ind.loc[A2, L_GEN])} · "
              f"ajuste de precio de mercado {pct(ind.loc[A1, L_AJ])} → {pct(ind.loc[A2, L_AJ])} · "
              f"fecha corta {pct(ind.loc[A1, L_FC])} → {pct(ind.loc[A2, L_FC])}")
    d = lamina(f"{nombre}: {frase_rango(c)}", bajada, seccion)
    imagen(d, figura_rangos(c, f"% de la venta bruta por rango de % cumplimiento del precio de lista general y motivo — "
                               f"{nombre} (al lado de cada barra: % y ajuste general, y el % de cada motivo)",
                            figsize=(14, 7.4), detalle=True))


def lamina_resumen_dim(dim, vista, soc, seccion):
    """Dos tablas: arriba volumen y pesos, abajo los tres % aplicados con sus ajustes."""
    res = resumen_dim(dim, soc, COLS_RES)
    mayor = res[(L_GEN, str(A2))].sub(res[(L_GEN, str(A1))]).idxmin()
    compacta = len(res) > 5                       # muchas filas: todo mas arriba y letra mas pequeña
    d = lamina(f"{dim}: {mayor} tuvo la mayor caída del % cumplimiento del precio de lista general "
               f"({pct(res.loc[mayor, (L_GEN, str(A1))])} → {pct(res.loc[mayor, (L_GEN, str(A2))])})",
               f"{vista} · venta kg, venta bruta y los tres % de cumplimiento del precio de lista con sus ajustes — "
               f"{PERIODO} {A1} vs {A2}", seccion, compacta=compacta)
    alto, y_ = Inches(0.3), Inches(1.2 if compacta else 1.6)
    for cols_, ancho, a_idx, tam in (([VENTA_KG, VENTA_BRUTA, P_GEN], Inches(10.0), 0.22, 10 if compacta else 10.5),
                                     (PILAR, UTIL_W, 0.16, 10 if compacta else 11)):
        sub = res[[col for col in res.columns if col[0] in cols_]]
        txt = a_texto(sub)
        txt.columns = pd.MultiIndex.from_tuples([(corto(i), a_) for i, a_ in sub.columns])
        txt.index = [str(i)[:32] for i in txt.index]
        txt.index.name = dim
        calor = None
        if dim in CON_CALOR:
            calor = sub[[col for col in sub.columns if col[0].startswith("% Ajuste")]].reindex(columns=sub.columns)
        tabla(d, txt, y=y_, w=ancho, tam=tam, ancho_indice=a_idx, alto_fila=alto, calor=calor)
        y_ += alto * (len(txt) + 2) + Inches(0.2)

## C.6 Informe de avance por región: definiciones

Las funciones del informe de avance que reutiliza el informe diario.

In [12]:
def nombre_corte(prefijo: str, vista: str) -> str:
    return (f"{prefijo}_{ARCHIVO[vista]}_{ETQ_ANT.replace(' ', '')}_vs_"
            f"{MES_ACT_TXT[:3].capitalize()}{INICIO_ACT.year}_al_{FECHA_MAX_D.day}.pptx")


def portada_corte(titulo: str, linea_2: str, soc):
    portada(titulo, f"{MES_ANT_TXT.capitalize()} {INICIO_ANT.year} (cerrado) vs {MES_ACT_TXT} al {FECHA_MAX_D.day} "
                    f"(en curso) · {nombre_sociedad(soc)}", f"{linea_2} · datos a {FECHA_MAX_D:%d/%m/%Y}",
            COLOR_SOC[soc])


def figura_tendencia_avance(soc, figsize=(15, 7.6), titulo=None):
    """Por dia (mes anterior y mes en curso): arriba el precio de lista y el precio neto por kg (lineas); abajo la venta
    neta (barras) con el % cumplimiento del precio de lista general (linea, eje derecho)."""
    t = indicadores(diario_de(soc)).reindex(DIAS_EJE)
    t.index = range(len(DIAS_EJE))
    color, tinte = COLOR_SOC[soc], TINTE_SOC[soc]
    corte = sum(DIAS_EJE.month == INICIO_ANT.month) - 0.5
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, sharex=True)

    def rotular(ax, s, formato, arriba=True):
        for x_ in (s.index[[0, -1]] if len(s) else []):   # primer y ultimo dia con dato
            ax.annotate(formato(s[x_]), (x_, s[x_]), xytext=(0, 6 if arriba else -10), textcoords="offset points",
                        ha="center", va="bottom" if arriba else "top", fontsize=7, color="#0b0b0b")

    for col, estilo, nombre, arriba in ((PRECIO_LISTA, "-", "Precio de lista por kg", True),
                                        (PRECIO_NETO, "--", "Precio neto por kg", False)):
        s = t[col].dropna()   # los dias sin venta no cortan la linea
        ax1.plot(s.index, s, color=color, linewidth=2, linestyle=estilo, marker="o", markersize=3.5, label=nombre)
        rotular(ax1, s, pesos, arriba)
    ejes_limpios(ax1, formato_y=lambda v, _: pesos(v))
    ax1.margins(y=0.18)
    ax1.set_title("Precio de lista y precio neto por kg, por día", loc="left", fontsize=10)
    ax1.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0))

    ax2.bar(t.index, t[VENTA_NETA], color=tinte, width=0.75, label="Venta neta ($)")
    ejes_limpios(ax2, formato_y=lambda v, _: "$" + num(v, 0))
    ax3 = ax2.twinx()
    s = t[L_GEN].dropna()
    ax3.plot(s.index, s, color=color, linewidth=2, marker="o", markersize=3.5, label="% cumplimiento (eje derecho)")
    rotular(ax3, s, pct)
    ax3.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}%")
    ax3.tick_params(colors="#52514e", labelsize=8)
    ax3.spines["top"].set_visible(False)
    ax3.margins(y=0.25)
    ax2.set_title("Venta neta por día ($, barras) y % cumplimiento del precio de lista general (línea)", loc="left",
                  fontsize=10)
    barras, lineas = ax2.get_legend_handles_labels(), ax3.get_legend_handles_labels()
    ax2.legend(barras[0] + lineas[0], barras[1] + lineas[1], frameon=False, fontsize=8, loc="upper left",
               bbox_to_anchor=(1.04, 1.0))
    for ax in (ax1, ax2):
        ax.axvline(corte, color="#8c8b87", linestyle=":", linewidth=1)
    ax2.set_xticks(range(len(DIAS_EJE)), DIAS_EJE.strftime("%d/%m"), rotation=90)
    fig.suptitle(f"{titulo or CORTO[soc]}: evolución día a día ({ETQ_ANT} y {ETQ_ACT})", x=0.01, ha="left", fontsize=12)
    plt.tight_layout()
    return fig

## C.8 Informe diario por canal de venta

`solo_canal_venta(cv)` deja las tablas del mes solo con ese canal; dentro, `modo_corte()` pone los periodos y las
láminas se arman con las mismas funciones del informe de avance.

In [13]:
@contextmanager
def solo_canal_venta(cv):
    """Mientras dura el `with`, las tablas del mes (diario_soc, cubo_d y la que usa modo_corte) solo traen el canal
    de venta cv. Va por fuera de modo_corte()."""
    g = globals()
    antes = {"diario_soc": diario_soc, "cubo_d": cubo_d}
    antes_cubo = CAMBIOS_CORTE["cubo"]
    g["diario_soc"] = diario_cv.xs(cv, level="Canal de Venta")
    g["cubo_d"] = CAMBIOS_CORTE["cubo"] = cubo_cv.xs(cv, level="Canal de Venta")
    try:
        yield
    finally:
        g.update(antes)
        CAMBIOS_CORTE["cubo"] = antes_cubo


def informe_diario(vista: str, cv: str):
    """Las laminas 1, 2, 3, 4, 5 y 14 del informe de avance, solo con el canal de venta cv."""
    soc = VISTAS[vista]
    nombre = f"{vista} · {cv}"
    with solo_canal_venta(cv), modo_corte():
        nueva_presentacion()
        # 1. portada
        portada_corte("Diagnóstico de Ajustes y % Cumplimiento Aplicado a Lista de Precios",
                      f"Canal de venta {cv} · avance diario", soc)
        # 2. panorama
        pan = a_texto(PANORAMA[vista], filas_indicador=True)
        pan.index.name = "Indicador"
        d = lamina(f"La lista por kilo {sube_baja(ch_(vista, PRECIO_LISTA))} {num(abs(ch_(vista, PRECIO_LISTA)))}%, "
                   f"y el neto por kilo {sube_baja(ch_(vista, PRECIO_NETO))} {num(abs(ch_(vista, PRECIO_NETO)))}%",
                   f"{nombre} · % cumplimiento del precio de lista general {de_a(vista, L_GEN)} · ajuste de precio "
                   f"de mercado {de_a(vista, L_AJ)} · fecha corta {de_a(vista, L_FC)}", sec(2))
        tabla(d, pan, x=Inches(1.5), w=Inches(10.3), y=Inches(1.5), tam=12, ancho_indice=0.46, alto_fila=Inches(0.28),
              negrita_filas=[PRECIO_LISTA, PRECIO_NETO, L_GEN, L_AJ, L_FC, KG_DIA, BRUTA_DIA])
        # 3. tendencia dia a dia
        d = lamina(seccion=sec(2))
        imagen(d, figura_tendencia_avance(soc, titulo=nombre), y=Inches(0.8), h=Inches(6.2))
        # 4. rangos del total
        lamina_rangos("Total", nombre, soc, "Total", sec(3))
        # 5. y 6. resumen por canal de distribucion y por grupo de material 6
        lamina_resumen_dim("Canal de distribución", nombre, soc, sec(4))
        lamina_resumen_dim("Grupo de material 6", nombre, soc, sec(6))
        guardar(nombre_corte("Informe_Diario", vista).replace(ARCHIVO[vista], f"{ARCHIVO[vista]}_{cv.replace(' ', '_')}"))


for vista in VISTAS:
    for cv in CANALES_VENTA_INFORME:
        informe_diario(vista, cv)

Informe_Diario_Region_Oriente_Institucional_Sep2026_vs_Oct2026_al_22.pptx: 6 láminas
Informe_Diario_Region_Oriente_Amas_de_casa_Sep2026_vs_Oct2026_al_22.pptx: 6 láminas
Informe_Diario_Region_Oriente_Modernos_Sep2026_vs_Oct2026_al_22.pptx: 6 láminas
Informe_Diario_Region_Oriente_Tradicional_Sep2026_vs_Oct2026_al_22.pptx: 6 láminas
Informe_Diario_Region_Occidente_Institucional_Sep2026_vs_Oct2026_al_22.pptx: 6 láminas
Informe_Diario_Region_Occidente_Amas_de_casa_Sep2026_vs_Oct2026_al_22.pptx: 6 láminas
Informe_Diario_Region_Occidente_Modernos_Sep2026_vs_Oct2026_al_22.pptx: 6 láminas
Informe_Diario_Region_Occidente_Tradicional_Sep2026_vs_Oct2026_al_22.pptx: 6 láminas
